> 在线讲义：[第 2 章 · 两个成功率比一比：差 5 个点、差 20 个点，哪个算数](https://wikieden.github.io/math-and-eval-stats/lessons/02-two-proportions.html) · 本书《数学与实验统计》

# 第 2 章 · 两个成功率比一比

讲义：[02-two-proportions.md](https://wikieden.github.io/math-and-eval-stats/lessons/02-two-proportions.html)。一格一格往下跑；只算讲义里讲过的例子，练习的数自己算。

## 讲义 ↔ 代码

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 | SE_diff、Wald、Newcombe | `diff_se / diff_wald_ci / newcombe_ci` | 14/20 vs 13/20：0.148、[−0.240, +0.340] |
| 第二步 | 合并 p 的 z 检验 | `two_proportion_z` | z 0.338、p 0.736；40/50 vs 30/50：p 0.029 |
| 第三步 | Fisher | `fisher_p` | 1.000；0.049 |
| 第四步 | 两个 Wilson 区间 | `wilson_ci` | [0.670, 0.888]、[0.462, 0.724] |

## 定义

In [ ]:
import math
import numpy as np
from scipy.stats import fisher_exact, norm
Z95 = 1.959964

`wilson_ci`：来自 `ch01_success_rate_ci.py`（展开在这里，notebook 单独能跑）

In [ ]:
def wilson_ci(k: int, n: int, z: float = Z95) -> tuple[float, float]:
    """第三步：Wilson 区间。中心往 0.5 拉一点，宽度里多一项 z²/4n²，所以 p̂=0 或 1 时也不会缩成一个点。"""
    p, z2 = k / n, z * z
    denom = 1 + z2 / n
    center = (p + z2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / denom
    return center - half, center + half

`diff_se`：第一步：差值的标准误 sqrt(p1(1-p1)/n1 + p2(1-p2)/n2)（两组独立，方差相加）。

In [ ]:
def diff_se(k1: int, n1: int, k2: int, n2: int) -> float:
    """第一步：差值的标准误 sqrt(p1(1-p1)/n1 + p2(1-p2)/n2)（两组独立，方差相加）。"""
    p1, p2 = k1 / n1, k2 / n2
    return math.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)

`diff_wald_ci`：第一步：差值的 Wald 区间 (p1 - p2) ± z·SE。

In [ ]:
def diff_wald_ci(k1, n1, k2, n2, z=Z95) -> tuple[float, float]:
    """第一步：差值的 Wald 区间 (p1 - p2) ± z·SE。"""
    d = k1 / n1 - k2 / n2
    se = diff_se(k1, n1, k2, n2)
    return d - z * se, d + z * se

`newcombe_ci`：第一步：Newcombe 区间（用两个 Wilson 区间拼出差值区间，小样本更稳）。

In [ ]:
def newcombe_ci(k1, n1, k2, n2, z=Z95) -> tuple[float, float]:
    """第一步：Newcombe 区间（用两个 Wilson 区间拼出差值区间，小样本更稳）。"""
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson_ci(k1, n1, z)
    l2, u2 = wilson_ci(k2, n2, z)
    d = p1 - p2
    lo = d - math.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)
    hi = d + math.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)
    return lo, hi

`two_proportion_z`：第二步：双比例 z 检验。零假设下两组成功率相同，用合并的 p 算标准误。返回 (z, 双侧 p 值)。

In [ ]:
def two_proportion_z(k1, n1, k2, n2) -> tuple[float, float]:
    """第二步：双比例 z 检验。零假设下两组成功率相同，用合并的 p 算标准误。返回 (z, 双侧 p 值)。"""
    pooled = (k1 + k2) / (n1 + n2)
    se0 = math.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    z = (k1 / n1 - k2 / n2) / se0
    return z, 2 * norm.sf(abs(z))

`fisher_p`：第三步：Fisher 精确检验的双侧 p 值（不靠正态近似，小样本用它）。

In [ ]:
def fisher_p(k1, n1, k2, n2) -> float:
    """第三步：Fisher 精确检验的双侧 p 值（不靠正态近似，小样本用它）。"""
    return fisher_exact([[k1, n1 - k1], [k2, n2 - k2]])[1]

`simulate_null_p`：第二步的模拟版：假设两组真成功率都等于合并成功率，重复做实验，数 |差值| ≥ 观测值的比例。

In [ ]:
def simulate_null_p(k1, n1, k2, n2, reps=200_000, seed=0) -> float:
    """第二步的模拟版：假设两组真成功率都等于合并成功率，重复做实验，数 |差值| ≥ 观测值的比例。"""
    rng = np.random.default_rng(seed)
    pooled = (k1 + k2) / (n1 + n2)
    d_obs = abs(k1 / n1 - k2 / n2)
    d = rng.binomial(n1, pooled, reps) / n1 - rng.binomial(n2, pooled, reps) / n2
    return float(np.mean(np.abs(d) >= d_obs - 1e-12))

## 一步一步跑

### 第一步 差值与区间（14/20 vs 13/20）

In [ ]:
diff_se(14, 20, 13, 20), diff_wald_ci(14, 20, 13, 20), newcombe_ci(14, 20, 13, 20)

### 第二步 z 检验与模拟 p 值

In [ ]:
two_proportion_z(14, 20, 13, 20), two_proportion_z(40, 50, 30, 50), simulate_null_p(40, 50, 30, 50)

### 第三步 Fisher 精确检验

In [ ]:
fisher_p(14, 20, 13, 20), fisher_p(40, 50, 30, 50)

### 第四步 区间重叠 ≠ 不显著

In [ ]:
wilson_ci(40, 50), wilson_ci(30, 50), two_proportion_z(40, 50, 30, 50)[1]

## 代码里才看得到的

1. z 检验用合并 p，区间用各自的 p̂，两个标准误不一样。
2. n 小时差值是离散的，模拟 p 值把等于观测值的情况也算进去，比 z 检验大。
3. fisher_exact 的表每行是 [成, 败]，不是 [成, 总数]。

## 自己改一改

- 80/100 vs 60/100 的 p 值
- 140/200 vs 130/200 的 p 值

In [ ]:
# 在这里改参数试试


## 口诀

> 两秤称重误差叠，差值区间跨零别下结论；
> 无罪推定先当一样，p 值问的是有多罕见；
> 小样本用 Fisher 算，区间重叠不算数；
> 差值、区间、p 值，三样一起报。